# 03 - VHI Crop Health Monitoring

Computes **VCI** (from MODIS NDVI vs 2001-present climatology), **TCI** (from MODIS LST), and **VHI = 0.5*VCI + 0.5*TCI**. Produces fortnightly state time-series, drought/heat stress classification (VHI < 40), and a heat-stress hotspot overlay.

In [ ]:
import sys, yaml
sys.path.append('..')
import ee, pandas as pd
from src import gee_utils, indices, visualization

with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)
gee_utils.init_ee()
SEASON_YEAR = 2023
VHI_CFG = cfg['vhi']

In [ ]:
# Multi-year climatology (2001 -> season start) for min/max normalisation
states_fc = gee_utils.get_states_fc(cfg['states'])
aoi = states_fc.geometry()
clim_start = f"{VHI_CFG['climatology_start_year']}-01-01"
ndvi_clim = gee_utils.get_modis_ndvi(aoi, clim_start, f'{SEASON_YEAR}-10-31')
lst_clim = gee_utils.get_modis_lst(aoi, clim_start, f'{SEASON_YEAR}-10-31')
print('Climatology images - NDVI:', ndvi_clim.size().getInfo(), '| LST:', lst_clim.size().getInfo())

## Fortnightly VHI per state

In [ ]:
start = ee.Date(f'{SEASON_YEAR}-11-01')
n_fortnights = 12  # Nov -> Apr
rows = []
for i in range(n_fortnights):
    f0 = start.advance(i * 15, 'day')
    f1 = f0.advance(15, 'day')
    ndvi_now = gee_utils.get_modis_ndvi(aoi, f0, f1).mean()
    lst_now = gee_utils.get_modis_lst(aoi, f0, f1).mean()
    vci = indices.compute_vci(ndvi_now, ndvi_clim)
    tci = indices.compute_tci(lst_now, lst_clim)
    vhi = indices.compute_vhi(vci, tci, VHI_CFG['alpha'])
    for st in cfg['states']:
        g = gee_utils.get_state_geometry(st)
        val = indices.regional_mean(vhi, g).get('VHI').getInfo()
        rows.append({'date': f0.format('YYYY-MM-dd').getInfo(), 'state': st, 'VHI': val})
vhi_df = pd.DataFrame(rows)
vhi_df['date'] = pd.to_datetime(vhi_df['date'])
vhi_df.to_csv('../outputs/vhi_fortnightly.csv', index=False)
vhi_df.head()

In [ ]:
visualization.vhi_timeseries(vhi_df, stress_threshold=VHI_CFG['stress_threshold'])

## Heat stress anomaly hotspot overlay
LST anomaly vs climatology mean, masked to stressed pixels (VHI < 40). Operationally, fuse with **IMD real-time temperature forecasts** (`src.io_utils.load_imd_netcdf`) to anticipate terminal heat stress in Feb-Mar.

In [ ]:
# Latest fortnight
f0 = start.advance((n_fortnights - 1) * 15, 'day')
f1 = f0.advance(15, 'day')
ndvi_now = gee_utils.get_modis_ndvi(aoi, f0, f1).mean()
lst_now = gee_utils.get_modis_lst(aoi, f0, f1).mean()
vhi_now = indices.compute_vhi(indices.compute_vci(ndvi_now, ndvi_clim),
                              indices.compute_tci(lst_now, lst_clim), VHI_CFG['alpha'])
lst_anom = lst_now.subtract(lst_clim.mean()).rename('LST_anom')
hotspots = lst_anom.updateMask(indices.vhi_stress_mask(vhi_now, VHI_CFG['stress_threshold']))

import geemap
m = geemap.Map(center=[26.5, 78.0], zoom=5)
m.addLayer(vhi_now.clip(aoi), {'min': 0, 'max': 100, 'palette': ['red', 'yellow', 'green']}, 'VHI')
m.addLayer(hotspots.clip(aoi), {'min': 0, 'max': 8, 'palette': ['orange', 'darkred']}, 'Heat stress hotspots')
m.addLayer(states_fc.style(color='black', fillColor='00000000'), {}, 'States')
m

In [ ]:
# Fortnightly bulletin summary table (latest fortnight, per state)
latest = vhi_df[vhi_df.date == vhi_df.date.max()].copy()
latest['condition'] = pd.cut(latest.VHI, [0, VHI_CFG['severe_threshold'], VHI_CFG['stress_threshold'], 100],
                             labels=['Severe stress', 'Stressed', 'Normal/Good'])
latest.sort_values('VHI')